<a href="https://colab.research.google.com/github/soundcatchers/Piper-Colab/blob/main/notebooks/piper_inference_(ckpt).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) inferencing notebook.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)

---

- Notebook made by [rmcpantoja](http://github.com/rmcpantoja)
- Collaborator: [Xx_Nessu_xX](https://github.com/XxNessuxX/)

# <font color="ffc800">🏁 ***First steps.*** 🏁

In [7]:
#@markdown # **Download model and config.** 📩
#@markdown ---
#@markdown ### Repairing legacy Numpy version locking and copying your step 6000 assets.

import os
import glob
from google.colab import drive

# 1. Mount Drive safely
if not os.path.exists("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

print("🔧 Repairing local dependencies and forcing Numpy 1.x compatibility...")
# 2. Force wipe the broken Numpy 2.x and lock down legacy compatibility
get_ipython().system('pip uninstall -y numpy')
get_ipython().system('pip install -q numpy==1.26.4 pytorch-lightning==1.9.0 torchmetrics==0.11.4 piper-phonemize-fix')

# 3. Structural path pointer setup
export_path = "/content/piper/src/python"
if not os.path.exists(export_path):
    print("🔄 Pulling repository directories into the environment path...")
    get_ipython().system('git clone -q https://github.com/rmcpantoja/piper')
    get_ipython().system('cd /content/piper/src/python && bash build_monotonic_align.sh')

# 4. Target your fresh, beautiful step 6000 files!
drive_folder = "/content/drive/MyDrive/Piper_Training/rick_sanchez"
checkpoint_files = glob.glob(os.path.join(drive_folder, "**", "*.ckpt"), recursive=True)

if not checkpoint_files:
    raise Exception("❌ File error: Could not see the checkpoints. Try hitting the refresh icon on the Colab sidebar.")

# Grab the absolute latest file (your step 6000 save!)
latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
config_file = os.path.join(drive_folder, "config.json")

# 5. Mirror them into local workspace memory
get_ipython().system('rm -rf /content/test_model && mkdir -p /content/test_model')
get_ipython().system(f'cp -f "{latest_checkpoint}" /content/test_model/model.ckpt')
get_ipython().system(f'cp -f "{config_file}" /content/test_model/config.json')

print("\n🎉 ENVIRONMENT REPAIRED & BOTH FILES COPIED!")
print(f"Loaded Checkpoint: {os.path.basename(latest_checkpoint)}")

🔧 Repairing local dependencies and forcing Numpy 1.x compatibility...
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires nu

In [8]:
#@markdown # **Download model and config.** 📩
import os
from google.colab import drive

if not os.path.exists("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

print("🔍 Bypassing Python cache... Asking the Linux OS to find the checkpoint directly:")

# Use the Linux terminal 'find' command to find any .ckpt files inside your project folder
# This completely ignores Google Drive's directory cache bugs!
found_checkpoints = !find "/content/drive/MyDrive/Piper_Training/rick_sanchez" -name "*.ckpt"

# Filter out empty entries or error lines
checkpoint_paths = [line for line in found_checkpoints if line.startswith("/content/drive") and ".ckpt" in line]

if not checkpoint_paths:
    print("❌ Linux couldn't find it either. Let's list the root folder contents to see what's happening:")
    !ls -R "/content/drive/MyDrive/Piper_Training/rick_sanchez"
    raise Exception("Could not locate the checkpoint. Check the folder printout above to see where the files are sitting.")
else:
    # Pick the first found checkpoint file
    latest_checkpoint = checkpoint_paths[0]
    config_file = "/content/drive/MyDrive/Piper_Training/rick_sanchez/config.json"

    print(f"\n🎯 Linux successfully found your file at: {latest_checkpoint}")
    print("Copying assets over to local testing environment...")

    !rm -rf /content/test_model && mkdir -p /content/test_model
    !cp -f "{latest_checkpoint}" /content/test_model/model.ckpt
    !cp -f "{config_file}" /content/test_model/config.json

    print("\n🎉 BOTH FILES COPIED SUCCESSFULLY!")
    print("Your environment is now perfectly locked down. Move to the inference cell!")

🔍 Bypassing Python cache... Asking the Linux OS to find the checkpoint directly:

🎯 Linux successfully found your file at: /content/drive/MyDrive/Piper_Training/rick_sanchez/lightning_logs/version_2/checkpoints/last.ckpt
Copying assets over to local testing environment...

🎉 BOTH FILES COPIED SUCCESSFULLY!
Your environment is now perfectly locked down. Move to the inference cell!


# <font color="ffc800">🔊 ***Inference.*** 🔊

In [9]:
#@markdown # <font color="ffc800">**Run inference.** 🤖
#@markdown ---
#@markdown #### Before you enjoy... <font color="orange">**Some notes!**
#@markdown * <font color="orange">You can run the cell to download voice packs and download voices you want at any time, even if you run this cell!</font>
#@markdown * <font color="orange">When you download a new voice, run this cell again and you will now be able to toggle between all the ones you download. Incredible, right?

#@markdown Enjoy!!

%cd /content/piper/src/python
# original: infer.py
import json
import logging
import sys
from pathlib import Path
from enum import Enum
from typing import Iterable, List, Optional, Union
import torch
from piper_train.vits.lightning import VitsModel
from piper_train.vits.utils import audio_float_to_int16
from piper_train.vits.wavfile import write as write_wav
import numpy as np
import glob
import ipywidgets as widgets
from IPython.display import display, Audio, Markdown, clear_output
from piper_phonemize import phonemize_codepoints, phonemize_espeak, tashkeel_run
import time
from pydub import AudioSegment
import scipy

_LOGGER = logging.getLogger("piper_train.infer_onnx")

def detect_ckpt_models(path):
    ckpt_models = glob.glob(path + '/*.ckpt')
    if len(ckpt_models) > 1:
        return ckpt_models
    elif len(ckpt_models) == 1:
        return ckpt_models[0]
    else:
        return None

def wav2mp3(path_to_file):
    final_audio = AudioSegment.from_wav(file=path_to_file)
    path_to_file = path_to_file.replace(".wav",".mp3")
    final_audio.export(path_to_file, format="mp3")
    return path_to_file

def main():
    """Main entry point"""
    models_path = "/content/piper/src/python"
    logging.basicConfig(level=logging.DEBUG)
    model = None
    ckpt_models = detect_ckpt_models(models_path)
    speaker_selection = widgets.Dropdown(
        options=[],
        description=f'{lan.translate(lang, "Select speaker")}:',
        layout={'visibility': 'hidden'}
    )
    if ckpt_models is None:
        if enhanced_accessibility:
            playaudio("novoices")
        raise Exception(lan.translate(lang, "No downloaded voice packages!"))
    elif isinstance(ckpt_models, str):
        ckpt_model = ckpt_models
        model, config = load_ckpt(ckpt_model)
        if config["num_speakers"] > 1:
            speaker_selection.options = config["speaker_id_map"].values()
            speaker_selection.layout.visibility = 'visible'
            preview_sid = 0
            if enhanced_accessibility:
                playaudio("multispeaker")
        else:
            speaker_selection.layout.visibility = 'hidden'
            preview_sid = None

        if enhanced_accessibility:
            inferencing(
                model,
                config,
                preview_sid,
                lan.translate(
                    config["espeak"]["voice"][:2],
                    "Interface openned. Write your texts, configure the different synthesis options or download all the voices you want. Enjoy!"
                )
            )
    else:
        voice_model_names = []
        for current in ckpt_models:
            voice_struct = current.split("/")[5]
            voice_model_names.append(voice_struct)
        if enhanced_accessibility:
            playaudio("selectmodel")
        selection = widgets.Dropdown(
            options=voice_model_names,
            description=f'{lan.translate(lang, "Select voice package")}:',
        )
        load_btn = widgets.Button(
            description=lan.translate(lang, "Load it!")
        )
        config = None
        def load_model(button):
            nonlocal config
            global ckpt_model
            nonlocal model
            nonlocal models_path
            selected_voice = selection.value
            ckpt_model = f"{models_path}/{selected_voice}"
            model, config = load_ckpt(ckpt_model)
            if enhanced_accessibility:
                playaudio("loaded")
            if config["num_speakers"] > 1:
                speaker_selection.options = config["speaker_id_map"].values()
                speaker_selection.layout.visibility = 'visible'
                if enhanced_accessibility:
                    playaudio("multispeaker")
            else:
                speaker_selection.layout.visibility = 'hidden'

        load_btn.on_click(load_model)
        display(selection, load_btn)
    display(speaker_selection)
    speed_slider = widgets.FloatSlider(
        value=1,
        min=0.25,
        max=4,
        step=0.1,
        description=lan.translate(lang, "Rate scale"),
        orientation='horizontal',
    )
    noise_scale_slider = widgets.FloatSlider(
        value=0.667,
        min=0.25,
        max=4,
        step=0.1,
        description=lan.translate(lang, "Phoneme noise scale"),
        orientation='horizontal',
    )
    noise_scale_w_slider = widgets.FloatSlider(
        value=1,
        min=0.25,
        max=4,
        step=0.1,
        description=lan.translate(lang, "Phoneme stressing scale"),
        orientation='horizontal',
    )
    sentence_silence  = widgets.Dropdown(
        options=[
            0.0,
            0.5,
            1.0,
            2.0,
            3.0,
            5.0
        ],
        value=0.0,
        description=lan.translate(lang, "Silence between sentences"),
    )
    play = widgets.Checkbox(
        value=True,
        description=lan.translate(lang, "Auto-play"),
        disabled=False
    )
    mp3 = widgets.Checkbox(
        value=False,
        description=lan.translate(lang, "Save to mp3"),
        disabled=False
    )
    text_input = widgets.Text(
        value='',
        placeholder=f'{lan.translate(lang, "Enter your text here")}:',
        description=lan.translate(lang, "Text to synthesize"),
        layout=widgets.Layout(width='80%')
    )
    synthesize_button = widgets.Button(
        description=lan.translate(lang, "Synthesize"),
        button_style='success', # 'success', 'info', 'warning', 'danger' or ''
        tooltip=lan.translate(lang, "Click here to synthesize the text."),
        icon='check'
    )
    close_button = widgets.Button(
        description=lan.translate(lang, "Exit"),
        tooltip=lan.translate(lang, "Closes this GUI."),
        icon='check'
    )

    def on_synthesize_button_clicked(b):
        if model is None:
            if enhanced_accessibility:
                playaudio("nomodel")
            raise Exception(lan.translate(lang, "You have not loaded any model from the list!"))
        text = text_input.value
        if config["num_speakers"] > 1:
            sid = speaker_selection.value
        else:
            sid = None
        rate = speed_slider.value
        noise_scale = noise_scale_slider.value
        noise_scale_w = noise_scale_w_slider.value
        silence = sentence_silence.value
        auto_play = play.value
        savemp3 = mp3.value
        inferencing(
            model,
            config,
            sid,
            text,
            rate,
            noise_scale,
            noise_scale_w,
            silence,
            auto_play,
            savemp3
        )

    def on_close_button_clicked(b):
        clear_output()
        if enhanced_accessibility:
            playaudio("exit")

    synthesize_button.on_click(on_synthesize_button_clicked)
    close_button.on_click(on_close_button_clicked)
    display(text_input)
    display(speed_slider)
    display(noise_scale_slider)
    display(noise_scale_w_slider)
    display(sentence_silence)
    display(play)
    display(mp3)
    display(synthesize_button)
    display(close_button)

def load_ckpt(model):
    _LOGGER.debug("Loading model from %s", model)
    config = load_config(model)
    model = VitsModel.load_from_checkpoint(str(model), dataset=None)
    # Inference only
    model.eval()
    with torch.no_grad():
        model.model_g.dec.remove_weight_norm()

    _LOGGER.info("Loaded model from %s", model)
    return model, config

def load_config(model):
    with open("config.json", "r") as file:
        config = json.load(file)
    return config

PAD = "_"  # padding (0)
BOS = "^"  # beginning of sentence
EOS = "$"  # end of sentence

class PhonemeType(str, Enum):
    ESPEAK = "espeak"
    TEXT = "text"

def phonemize(config, text: str) -> List[List[str]]:
    """Text to phonemes grouped by sentence."""
    if config["phoneme_type"] == PhonemeType.ESPEAK:
        if config["espeak"]["voice"] == "ar":
            # Arabic diacritization
            # https://github.com/mush42/libtashkeel/
            text = tashkeel_run(text)
        return phonemize_espeak(text, config["espeak"]["voice"])
    if config["phoneme_type"] == PhonemeType.TEXT:
        return phonemize_codepoints(text)
    raise ValueError(f"Unexpected phoneme type: {self.config.phoneme_type}")

def phonemes_to_ids(config, phonemes: List[str]) -> List[int]:
    """Phonemes to ids."""
    id_map = config["phoneme_id_map"]
    ids: List[int] = list(id_map[BOS])
    for phoneme in phonemes:
        if phoneme not in id_map:
            print("Missing phoneme from id map: %s", phoneme)
            continue
        ids.extend(id_map[phoneme])
        ids.extend(id_map[PAD])
    ids.extend(id_map[EOS])
    return ids

def inferencing(model, config, sid, line, length_scale = 1, noise_scale = 0.667, noise_scale_w = 0.8, sentence_silence = 0.0, auto_play=True, save_mp3 = False):
    audios = []
    if config["phoneme_type"] == "PhonemeType.ESPEAK":
        config["phoneme_type"] = "espeak"
    text = phonemize(config, line)
    num_silence_samples = int(sentence_silence * config["audio"]["sample_rate"])
    silence = np.zeros(num_silence_samples, dtype=np.int16)
    for phonemes in text:
        phoneme_ids = phonemes_to_ids(config, phonemes)
        num_speakers = config["num_speakers"]
        if num_speakers == 1:
            speaker_id = None # for now
        else:
            speaker_id = sid
        text = torch.LongTensor(phoneme_ids).unsqueeze(0)
        text_lengths = torch.LongTensor([len(phoneme_ids)])
        scales = [
            noise_scale,
            length_scale,
            noise_scale_w
        ]
        sid = torch.LongTensor([speaker_id]) if speaker_id is not None else None
        audio = model(
            text,
            text_lengths,
            scales,
            sid=sid
        ).detach().numpy()
        audio = audio_float_to_int16(audio.squeeze())
        audio = np.concatenate((audio, silence))
        audios.append(audio)
    merged_audio = np.concatenate(audios)
    sample_rate = config["audio"]["sample_rate"]
    display(Markdown(f"{line}"))
    if save_mp3:
        timestr = time.strftime("%Y%m%d-%H%M%S")
        new_audio_name = f"{timestr}.wav"
        scipy.io.wavfile.write(new_audio_name, sample_rate, merged_audio)
        converted = wav2mp3(new_audio_name)
        display(Audio(converted, rate=sample_rate, autoplay=auto_play))
    else:
        display(Audio(merged_audio, rate=sample_rate, autoplay=auto_play))

def denoise(
    audio: np.ndarray, bias_spec: np.ndarray, denoiser_strength: float
) -> np.ndarray:
    audio_spec, audio_angles = transform(audio)

    a = bias_spec.shape[-1]
    b = audio_spec.shape[-1]
    repeats = max(1, math.ceil(b / a))
    bias_spec_repeat = np.repeat(bias_spec, repeats, axis=-1)[..., :b]

    audio_spec_denoised = audio_spec - (bias_spec_repeat * denoiser_strength)
    audio_spec_denoised = np.clip(audio_spec_denoised, a_min=0.0, a_max=None)
    audio_denoised = inverse(audio_spec_denoised, audio_angles)

    return audio_denoised


def stft(x, fft_size, hopsamp):
    """Compute and return the STFT of the supplied time domain signal x.
    Args:
        x (1-dim Numpy array): A time domain signal.
        fft_size (int): FFT size. Should be a power of 2, otherwise DFT will be used.
        hopsamp (int):
    Returns:
        The STFT. The rows are the time slices and columns are the frequency bins.
    """
    window = np.hanning(fft_size)
    fft_size = int(fft_size)
    hopsamp = int(hopsamp)
    return np.array(
        [
            np.fft.rfft(window * x[i : i + fft_size])
            for i in range(0, len(x) - fft_size, hopsamp)
        ]
    )


def istft(X, fft_size, hopsamp):
    """Invert a STFT into a time domain signal.
    Args:
        X (2-dim Numpy array): Input spectrogram. The rows are the time slices and columns are the frequency bins.
        fft_size (int):
        hopsamp (int): The hop size, in samples.
    Returns:
        The inverse STFT.
    """
    fft_size = int(fft_size)
    hopsamp = int(hopsamp)
    window = np.hanning(fft_size)
    time_slices = X.shape[0]
    len_samples = int(time_slices * hopsamp + fft_size)
    x = np.zeros(len_samples)
    for n, i in enumerate(range(0, len(x) - fft_size, hopsamp)):
        x[i : i + fft_size] += window * np.real(np.fft.irfft(X[n]))
    return x


def inverse(magnitude, phase):
    recombine_magnitude_phase = np.concatenate(
        [magnitude * np.cos(phase), magnitude * np.sin(phase)], axis=1
    )

    x_org = recombine_magnitude_phase
    n_b, n_f, n_t = x_org.shape  # pylint: disable=unpacking-non-sequence
    x = np.empty([n_b, n_f // 2, n_t], dtype=np.complex64)
    x.real = x_org[:, : n_f // 2]
    x.imag = x_org[:, n_f // 2 :]
    inverse_transform = []
    for y in x:
        y_ = istft(y.T, fft_size=1024, hopsamp=256)
        inverse_transform.append(y_[None, :])

    inverse_transform = np.concatenate(inverse_transform, 0)

    return inverse_transform


def transform(input_data):
    x = input_data
    real_part = []
    imag_part = []
    for y in x:
        y_ = stft(y, fft_size=1024, hopsamp=256).T
        real_part.append(y_.real[None, :, :])  # pylint: disable=unsubscriptable-object
        imag_part.append(y_.imag[None, :, :])  # pylint: disable=unsubscriptable-object
    real_part = np.concatenate(real_part, 0)
    imag_part = np.concatenate(imag_part, 0)

    magnitude = np.sqrt(real_part**2 + imag_part**2)
    phase = np.arctan2(imag_part.data, real_part.data)

    return magnitude, phase

main()

/content/piper/src/python


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# <font color="orange">**I suggest you export this model to use it in piper!**

Well, have you tried it yet? What do you think about it? If both answers are acceptable, it's time to disconnect your session in this notebook and [export this model](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_model_exporter.ipynb)!